In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
BATCH_SIZE = 4 # how many independent sequences will we process in parallel?
BLOCK_SIZE = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
N_EMB = 32
# ------------

torch.manual_seed(1337)

In [ ]:
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
VOCAB_SIZE = len(chars)

# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# Tokenize: raw text to a sequence of integers
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string
# More advanced tokenizers return shorter lists (we return i for each char), but with higher max i

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
len(data)

1115394

In [8]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i+1:i+BLOCK_SIZE+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        # eg 32x3 in
        # 32x3x10 out
        # for each of 3 chars we pull 10 embs
        self.token_embedding_table = nn.Embedding(VOCAB_SIZE, N_EMB)

        # Emb position
        # Each position, from 0 to BLOCK_SIZE-1, will get its own embedding vector
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMB)

        self.lm1 = nn.Linear(N_EMB, VOCAB_SIZE)

    def forward(self, x, targets=None):
        """
        Forward pass + Loss

        """

        # x and targets are both (B,T) tensor of integers
        # x is BatchSize x Time (block size)

        B, T = x.shape

        te = self.token_embedding_table(x) # (B,T,C) Batch(size=4) Time(=Block=8) Channel(=emb_dim=vocab size=65)
        pe = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) Time(=Block=8) Channel(=emb_dim=vocab size=65)

        e = te + pe # (B,T,C) add the position information to the token information
        
        logits = self.lm1(e) # (B,T,C2)

        if targets is None:
            loss = None
        else:
            # pyTorch cross_entropy want B, C, T but we have B, T, C so need to reshape
            # Batch
            # Time
            # Channel
            print(logits.shape)
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # preserve C, stretch out B and T into single dim
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        # T is Embeddings
        for _ in range(max_new_tokens):
            # get the predictions
            logits, _ = self.forward(idx)
            # focus only on the last time step - T is 1
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [11]:
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [ ]:
model = BigramLanguageModel(VOCAB_SIZE)
# m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


In [23]:
for iter in range(1):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss(model)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()



torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size([4, 8, 65])
torch.Size(

In [ ]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))

In [ ]:
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
print(x.shape)

In [39]:
torch.manual_seed(42)
_B, _T, _C = 4, 8, 10
# b= torch.randint(0, 10, (_B, _T, _C)).float() # (B, T, C)
b= torch.rand(_B, _T, _C).float() # (B, T, C)
tril = torch.tril(torch.ones(_T, _T))

a = tril / torch.sum(tril, dim=1, keepdim=True)
print(a)
# print(b)

# c is running mean of Embeddings, ACROSS _T, for each B
c = a @ b # 4x8x10
# print(c)
    
w = torch.zeros((_T, _T))
we = w.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(we, dim=-1)
print(wei)
c2 = wei @ b

assert torch.allclose(c, c2)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.16

In [ ]:
# Self Attention

torch.manual_seed(42)
_B, _T, _C = 4, 8, 10
b= torch.rand(_B, _T, _C).float() # (B, T, C)
tril = torch.tril(torch.ones(_T, _T))

# Single Head self-attention
hs = 16 # Head Size
key = nn.Linear(_C, hs, bias=False)
query = nn.Linear(_C, hs, bias=False)

# No Communication yet
k = key(b) # (B, T, C) -> (B, T, hs) - k is "what do I contain?"
q = query(b) # (B, T, hs) - q is "what am I looking for?"

# Now communicate
# dot product of queries and keys is a measure of how much 
# each position should attend to each other position
w = q @ k.transpose(-2, -1) # (B, T, hs) @ (B, hs, T) -> (B, T, T)
# First row (of q, dim hs) mult by first col of k(transposed) (dim hs)
# first row of q = mix (matrix mult) of b(x)
# first col of k.T = mix (matrix mult) of b(x)
# qs of T1 dot ks of T1, qs of T1 dot ks of T2, qs of T1 dot ks of T3, ...
# qs of T2 dot ks of T1, qs of T2 dot ks of T2, qs of T2 dot ks of T3, ...

# And now do running mean
print(w[0])
we = w.masked_fill(tril == 0, float('-inf'))
# We don't want eg 5th node to take information from 6th, 7th
print(we[0])
wei = F.softmax(we, dim=-1)
print(wei[0]) # rows sum to 1, but only over the first i cols, rest is 0
out = wei @ b # (B, T, T) @ (B, T, C) -> (B, T, C)
print(out.shape) # (B, T, C)

# 

tensor([[[-7.8197e-01, -2.1053e-01, -8.5854e-01,  2.8162e-01, -7.1930e-01,
           4.8790e-01,  4.5848e-01,  1.7307e-02, -3.9614e-01, -1.4393e-01,
          -2.4425e-01,  1.5659e-01,  7.2901e-01, -4.7151e-02, -5.0799e-02,
           1.9998e-01],
         [-6.9020e-01, -1.1475e-01, -4.5746e-01,  5.6526e-01, -4.8085e-01,
          -6.6705e-03,  1.4497e-01, -1.3667e-01, -4.3763e-01,  3.2981e-02,
          -1.6911e-01, -9.2299e-02,  1.9524e-01, -3.2100e-01, -6.4731e-02,
           2.4456e-01],
         [-2.4683e-01, -3.1235e-01, -4.3164e-01,  2.5514e-01, -3.3757e-01,
           2.0023e-01,  2.7404e-01,  4.5960e-02, -2.1980e-01, -5.5548e-02,
          -6.9632e-02,  8.2316e-02,  3.7099e-01, -9.2494e-02, -1.1562e-01,
           6.2080e-02],
         [-7.5382e-01, -2.6952e-01, -6.1469e-01,  4.1259e-01, -6.0907e-01,
           1.3358e-01,  4.9345e-01, -1.2915e-02, -6.5934e-01,  2.9528e-02,
          -3.2946e-01,  1.1946e-01,  1.9190e-01,  1.7360e-02, -1.8059e-01,
           2.1857e-01],
    

In [27]:
torch.__version__

'2.1.2+cpu'